In [ ]:
!pip install snntorch python-dotenv

In [ ]:
import snntorch as snn
from snntorch import spikeplot as splt
from snntorch import spikegen

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np
import itertools
import os

In [ ]:
# Load .env file if it exists
from dotenv import load_dotenv
load_dotenv()

# SETUP DEVICE
batch_size = int(os.environ.get("BATCH_SIZE", "8"))
IMG_SIZE = int(os.environ.get("IMG_SIZE", "416"))
GRID_SIZE = 13   # 416 / 32 = 13 after 5 max-pool layers
NUM_CLASSES = 2   # 0=human, 1=car
BBOX_CHANNELS = 4
OUT_CHANNELS = NUM_CLASSES + BBOX_CHANNELS + 1  # class(2) + bbox(4) + confidence(1) = 7
TEST_DATA = os.environ.get("TEST_DATA")
VALID_DATA = os.environ.get("VALID_DATA")
TRAIN_DATA = os.environ.get("TRAIN_DATA")

dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")
print(f"Train: {TRAIN_DATA}")
print(f"Test:  {TEST_DATA}")
print(f"Valid: {VALID_DATA}")

In [ ]:
# DATA PREPARATION -- Full-image detection dataset with grid targets
from PIL import Image

class DetectionDataset(torch.utils.data.Dataset):
    def __init__(self, data_dir, img_size=416, grid_size=13):
        self.samples = []
        for txt_file in sorted(os.listdir(data_dir)):
            if not txt_file.endswith('.txt'):
                continue
            txt_path = os.path.join(data_dir, txt_file)
            img_path = os.path.splitext(txt_path)[0] + '.jpg'
            if not os.path.exists(img_path):
                continue
            # Parse all objects for this image
            objects = []
            with open(txt_path, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parts = line.split()
                    values = [float(p) for p in parts]
                    label = int(values[0])
                    coords = values[1:]
                    for i in range(0, len(coords), 4):
                        bbox = coords[i:i+4]
                        if len(bbox) == 4:
                            objects.append((label, bbox))
            if objects:
                self.samples.append((img_path, objects))
        self.img_size = img_size
        self.grid_size = grid_size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, objects = self.samples[idx]
        img = Image.open(img_path).convert('L')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img_tensor = torch.tensor(np.array(img), dtype=torch.float32).unsqueeze(0) / 255.0

        # Build target tensor: [7, grid, grid]
        target = torch.zeros(OUT_CHANNELS, self.grid_size, self.grid_size, dtype=torch.float32)

        for label, bbox in objects:
            x_c, y_c, w, h = bbox  # normalized [0,1]
            gx = int(x_c * self.grid_size)
            gy = int(y_c * self.grid_size)
            gx = min(gx, self.grid_size - 1)
            gy = min(gy, self.grid_size - 1)

            # Skip if cell already occupied
            if target[6, gy, gx] > 0:
                continue

            # Class one-hot
            target[label, gy, gx] = 1.0
            target[1 - label, gy, gx] = 0.0
            # Bbox relative to grid cell
            cx = x_c * self.grid_size - gx
            cy = y_c * self.grid_size - gy
            cw = w * self.grid_size
            ch = h * self.grid_size
            target[2, gy, gx] = cx
            target[3, gy, gx] = cy
            target[4, gy, gx] = cw
            target[5, gy, gx] = ch
            # Confidence
            target[6, gy, gx] = 1.0

        return img_tensor, target


train_dataset = DetectionDataset(TRAIN_DATA, img_size=IMG_SIZE, grid_size=GRID_SIZE)
test_dataset = DetectionDataset(TEST_DATA, img_size=IMG_SIZE, grid_size=GRID_SIZE)
valid_dataset = DetectionDataset(VALID_DATA, img_size=IMG_SIZE, grid_size=GRID_SIZE)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, drop_last=False)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

print(f"Train images: {len(train_dataset)}, Test images: {len(test_dataset)}, Valid images: {len(valid_dataset)}")

In [ ]:
# SUPPORT FUNCTIONS
# ====================
# Loss Function

def detection_loss(pred, target):
    # pred, target: [batch, 7, grid, grid]
    obj_mask = target[:, 6:7, :, :]  # [batch, 1, grid, grid] -- confidence channel
    noobj_mask = 1.0 - obj_mask

    # Classification loss (MSE, only on cells with objects)
    cls_pred = pred[:, 0:2, :, :]
    cls_target = target[:, 0:2, :, :]
    cls_loss = (obj_mask * (cls_pred - cls_target) ** 2).sum() / (obj_mask.sum() + 1e-6)

    # Bbox regression loss (MSE, only on cells with objects)
    bbox_pred = pred[:, 2:6, :, :]
    bbox_target = target[:, 2:6, :, :]
    box_loss = (obj_mask * (bbox_pred - bbox_target) ** 2).sum() / (obj_mask.sum() + 1e-6)

    # Confidence loss
    conf_pred = pred[:, 6:7, :, :]
    conf_obj = (obj_mask * (conf_pred - 1.0) ** 2).sum() / (obj_mask.sum() + 1e-6)
    conf_noobj = (noobj_mask * (conf_pred - 0.0) ** 2).sum() / (noobj_mask.sum() + 1e-6)

    return cls_loss + 5.0 * box_loss + conf_obj + 0.5 * conf_noobj


# ====================
# IoU & Decoding

def compute_iou(box1, box2):
    """IoU between two boxes in YOLO normalized (x_c, y_c, w, h) format."""
    x_c1, y_c1, w1, h1 = box1
    x_c2, y_c2, w2, h2 = box2

    x1_min, y1_min = x_c1 - w1/2, y_c1 - h1/2
    x1_max, y1_max = x_c1 + w1/2, y_c1 + h1/2
    x2_min, y2_min = x_c2 - w2/2, y_c2 - h2/2
    x2_max, y2_max = x_c2 + w2/2, y_c2 + h2/2

    inter_w = max(0, min(x1_max, x2_max) - max(x1_min, x2_min))
    inter_h = max(0, min(y1_max, y2_max) - max(y1_min, y2_min))
    inter_area = inter_w * inter_h

    area1, area2 = w1 * h1, w2 * h2
    return inter_area / (area1 + area2 - inter_area + 1e-6)


def decode_predictions(pred, conf_threshold=0.0):
    """Decode a single prediction [7, grid, grid] into detections.
    Returns list of (class_id, confidence, x_c, y_c, w, h) in normalized coords.
    """
    detections = []
    for gy in range(GRID_SIZE):
        for gx in range(GRID_SIZE):
            conf = pred[6, gy, gx].item()
            if conf <= conf_threshold:
                continue
            class_id = pred[0:2, gy, gx].argmax().item()
            cx = pred[2, gy, gx].item()
            cy = pred[3, gy, gx].item()
            cw = pred[4, gy, gx].item()
            ch = pred[5, gy, gx].item()
            # Convert grid-relative to normalized image coords
            x_c = max(0, min(1, (gx + cx) / GRID_SIZE))
            y_c = max(0, min(1, (gy + cy) / GRID_SIZE))
            w = max(0, min(1, cw / GRID_SIZE))
            h = max(0, min(1, ch / GRID_SIZE))
            detections.append((class_id, conf, x_c, y_c, w, h))
    return detections


def decode_targets(target):
    """Decode a single target [7, grid, grid] into ground truth boxes.
    Returns list of (class_id, x_c, y_c, w, h) in normalized coords.
    """
    gt_boxes = []
    for gy in range(GRID_SIZE):
        for gx in range(GRID_SIZE):
            if target[6, gy, gx] > 0:
                class_id = target[0:2, gy, gx].argmax().item()
                cx = target[2, gy, gx].item()
                cy = target[3, gy, gx].item()
                cw = target[4, gy, gx].item()
                ch = target[5, gy, gx].item()
                x_c = (gx + cx) / GRID_SIZE
                y_c = (gy + cy) / GRID_SIZE
                w = cw / GRID_SIZE
                h = ch / GRID_SIZE
                gt_boxes.append((class_id, x_c, y_c, w, h))
    return gt_boxes


# ====================
# Network Factory

def build_network(layer_num, beta, step_num, hidden_num):
    """Create and return a DetectionNet on the configured device."""
    return DetectionNet(layer_num=layer_num, beta=beta,
                        step_num=step_num, hidden_num=hidden_num).to(device)

def build_optimizer(net, lr=1e-4):
    """Create an Adam optimizer for the network."""
    return torch.optim.Adam(net.parameters(), lr=lr, betas=(0.9, 0.999))


# ====================
# Evaluation

def evaluate_detection(net, loader, step_num, conf_threshold=0.0, iou_threshold=0.5):
    """Full detection evaluation with IoU matching.
    Returns (precision, recall, f1, per_class_stats).
    """
    all_gt = []
    all_pred = []

    with torch.no_grad():
        net.eval()
        for data, targets in loader:
            data = data.to(device)
            targets = targets.to(device)
            spike_data = spikegen.rate(data, num_steps=step_num)
            _, mem_rec = net(spike_data)
            preds = mem_rec.sum(dim=0)

            for i in range(preds.size(0)):
                all_gt.append(decode_targets(targets[i]))
                detections = decode_predictions(preds[i], conf_threshold)
                detections.sort(key=lambda x: x[1], reverse=True)
                all_pred.append(detections)

    per_class = {c: {'tp': 0, 'fp': 0, 'fn': 0} for c in range(NUM_CLASSES)}

    for gt_boxes, pred_boxes in zip(all_gt, all_pred):
        matched_gt = set()
        for pred_cls, conf, px, py, pw, ph in pred_boxes:
            best_iou = 0
            best_gt = -1
            for gt_idx, (gt_cls, gx, gy, gw, gh) in enumerate(gt_boxes):
                if gt_idx in matched_gt or pred_cls != gt_cls:
                    continue
                iou = compute_iou((px, py, pw, ph), (gx, gy, gw, gh))
                if iou > best_iou:
                    best_iou = iou
                    best_gt = gt_idx
            if best_iou >= iou_threshold:
                per_class[pred_cls]['tp'] += 1
                matched_gt.add(best_gt)
            else:
                per_class[pred_cls]['fp'] += 1

        for gt_idx, (gt_cls, _, _, _, _) in enumerate(gt_boxes):
            if gt_idx not in matched_gt:
                per_class[gt_cls]['fn'] += 1

    total_tp = sum(c['tp'] for c in per_class.values())
    total_fp = sum(c['fp'] for c in per_class.values())
    total_fn = sum(c['fn'] for c in per_class.values())

    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-6)

    for c in range(NUM_CLASSES):
        pc = per_class[c]
        pc['precision'] = pc['tp'] / max(pc['tp'] + pc['fp'], 1)
        pc['recall'] = pc['tp'] / max(pc['tp'] + pc['fn'], 1)

    return precision, recall, f1, per_class


# ====================
# Training

def train_one_epoch(net, train_loader, optimizer, step_num, epoch, print_every=10):
    """Run one training epoch. Returns list of loss values."""
    loss_hist = []
    for batch_idx, (data, targets) in enumerate(train_loader):
        data = data.to(device)
        targets = targets.to(device)

        spike_data = spikegen.rate(data, num_steps=step_num)

        net.train()
        spk_rec, mem_rec = net(spike_data)

        pred = mem_rec.sum(dim=0)
        loss_val = detection_loss(pred, targets)

        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()
        loss_hist.append(loss_val.item())

        if batch_idx % print_every == 0:
            train_printer(pred[0].detach(), targets[0], epoch, batch_idx, loss_hist)

    return loss_hist


def train_for_epochs(net, optimizer, train_loader, step_num, num_epochs):
    """Train network for num_epochs. Returns combined loss history."""
    all_losses = []
    for epoch in range(num_epochs):
        losses = train_one_epoch(net, train_loader, optimizer, step_num, epoch)
        all_losses.extend(losses)
    return all_losses


# ====================
# Logging

CLASS_NAMES = {0: 'human', 1: 'car'}

def print_config_results(beta, step_num, layer_num, hidden_num,
                         precision, recall, f1, per_class):
    """Print formatted results for a single hyperparameter config."""
    print(f"--- beta={beta}, steps={step_num}, layers={layer_num}, neurons={hidden_num} ---")
    print(f"Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}")
    for c in range(NUM_CLASSES):
        pc = per_class[c]
        print(f"  {CLASS_NAMES[c]:6s}: P={pc['precision']:.3f}  R={pc['recall']:.3f}  "
              f"TP={pc['tp']}  FP={pc['fp']}  FN={pc['fn']}")


def print_detection_metrics(pred, target):
    # pred, target: single item [7, grid, grid]
    obj_mask = target[6, :, :] > 0
    num_gt = int(obj_mask.sum().item())
    if num_gt == 0:
        return

    conf_pred = pred[6, :, :]
    cls_pred = pred[0:2, :, :]
    cls_target = target[0:2, :, :]

    _, pred_cls = cls_pred.max(dim=0)
    _, true_cls = cls_target.max(dim=0)
    cls_correct = (pred_cls[obj_mask] == true_cls[obj_mask]).sum().item()
    cls_acc = 100 * cls_correct / num_gt

    mean_conf = conf_pred[obj_mask].mean().item()

    print(f"    GT objects: {num_gt} | Class acc: {cls_acc:.1f}% | Mean obj conf: {mean_conf:.3f}")


def train_printer(pred, target, epoch, batch_idx, loss_hist):
    print(f"Epoch {epoch}, Batch {batch_idx}")
    print(f"Train Loss: {loss_hist[-1]:.4f}")
    print_detection_metrics(pred, target)
    print()

In [ ]:
# NETWORK DEFINITION
class DetectionNet(nn.Module):
    def __init__(self, layer_num, beta, step_num, hidden_num, stdp_lr=0.01, weight_decay=0.99, trace_decay=0.99):
        super().__init__()
        self.layer_num = layer_num
        self.beta = beta
        self.step_num = step_num
        self.hidden_num = hidden_num
        self.stdp_lr = stdp_lr
        self.weight_decay = weight_decay
        self.trace_decay = trace_decay

        num_inputs = IMG_SIZE * IMG_SIZE              # 416*416 = 173056
        num_outputs = OUT_CHANNELS * GRID_SIZE * GRID_SIZE  # 7*13*13 = 1183

        # initialize input, inner, and output layer
        self.inputFC = nn.Linear(num_inputs, self.hidden_num)
        self.inputLIF = snn.Leaky(beta=self.beta)
        self.innerLayer = nn.ModuleList()
        self.innerLIF = nn.ModuleList()
        for _ in range(layer_num):
            self.innerLayer.append(nn.Linear(self.hidden_num, self.hidden_num))
            self.innerLIF.append(snn.Leaky(beta=self.beta))
        self.outputFC = nn.Linear(self.hidden_num, num_outputs)
        self.outputLIF = snn.Leaky(beta=self.beta)

    def forward(self, spike_data):
        # spike_data: [step_num, batch, 1, 416, 416]
        mem_in = self.inputLIF.init_leaky()
        mem_hidden = [lif.init_leaky() for lif in self.innerLIF]
        mem_out = self.outputLIF.init_leaky()

        spk_rec, mem_rec = [], []

        for step in range(spike_data.size(0)):
            spk = spike_data[step]
            spk = spk.view(spk.size(0), -1)  # flatten: [batch, 173056]

            cur = self.inputFC(spk)
            spk, mem_in = self.inputLIF(cur, mem_in)
            for idx in range(self.layer_num):
                cur = self.innerLayer[idx](spk)
                spk, mem_hidden[idx] = self.innerLIF[idx](cur, mem_hidden[idx])

            cur = self.outputFC(spk)
            spk, mem_out = self.outputLIF(cur, mem_out)
            spk_rec.append(spk)
            mem_rec.append(mem_out)

        spk_rec = torch.stack(spk_rec, dim=0)  # [T, batch, 1183]
        mem_rec = torch.stack(mem_rec, dim=0)
        # Reshape to grid: [T, batch, 7, 13, 13]
        spk_rec = spk_rec.view(spk_rec.size(0), spk_rec.size(1), OUT_CHANNELS, GRID_SIZE, GRID_SIZE)
        mem_rec = mem_rec.view(mem_rec.size(0), mem_rec.size(1), OUT_CHANNELS, GRID_SIZE, GRID_SIZE)
        return spk_rec, mem_rec

In [ ]:
# NETWORK PARAMETERS
betas = [0.85, 0.95]
layer_nums = [1, 2]
neuron_nums = [64, 128]
step_nums = [25, 50]

In [ ]:
# TRAINING LOOP — Grid search over hyperparameters
num_epochs = 10
f1_rec = []

for beta in betas:
  for step_num in step_nums:
    for layer_num in layer_nums:
      for hidden_num in neuron_nums:

        # Step 1: Build the network and optimizer for this config
        net = build_network(layer_num, beta, step_num, hidden_num)
        optimizer = build_optimizer(net)

        # Step 2: Train the network
        loss_hist = train_for_epochs(net, optimizer, train_loader, step_num, num_epochs)

        # Step 3: Evaluate with IoU-based detection metrics
        precision, recall, f1, per_class = evaluate_detection(net, test_loader, step_num)

        # Step 4: Record and print results
        print_config_results(beta, step_num, layer_num, hidden_num, precision, recall, f1, per_class)
        f1_rec.append(f1)

print(f1_rec)

In [ ]:
# HEATMAP DATA PREPARATION
print(f"Total configurations tested: {len(f1_rec)}")
heatmap_data = np.array(f1_rec).reshape(len(layer_nums) * len(neuron_nums), len(betas) * len(step_nums))
print(heatmap_data)

In [ ]:
# HEATMAP PLOTTING
data_4d = np.array(f1_rec).reshape(len(betas), len(step_nums), len(layer_nums), len(neuron_nums))

# Permute dimensions: 0:Beta, 1:Step, 2:Layer, 3:Neuron -> 2:Layer, 3:Neuron, 0:Beta, 1:Step
data_permuted = data_4d.transpose(2, 3, 0, 1)
data_matrix = data_permuted.reshape(len(layer_nums) * len(neuron_nums), len(betas) * len(step_nums))

fig, ax = plt.subplots(figsize=(12, 8))

im = ax.imshow(data_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)

cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel("F1 Score", rotation=-90, va="bottom")

for i in range(data_matrix.shape[0]):
    for j in range(data_matrix.shape[1]):
        text = ax.text(j, i, f"{data_matrix[i, j]:.3f}",
                       ha="center", va="center", color="w", fontsize=9, fontweight='bold')

# X-AXIS (Beta / Step)
ax.set_xticks(np.arange(len(betas) * len(step_nums)))
ax.set_xticklabels(step_nums * len(betas))
ax.set_xlabel("Steps (Inner) / Beta (Outer)", labelpad=50, fontweight='bold', fontsize=12)

for i, b in enumerate(betas):
    center_x = (i * len(step_nums)) + (len(step_nums) - 1) / 2
    ax.text(center_x, data_matrix.shape[0] + 0.5, f"beta = {b}", ha='center', va='top', fontsize=11, fontweight='bold', color='navy')
    if i > 0:
        ax.axvline(x=(i * len(step_nums)) - 0.5, color='white', linewidth=2)

# Y-AXIS (Layer / Neuron)
ax.set_yticks(np.arange(data_matrix.shape[0]))
ax.set_yticklabels(neuron_nums * len(layer_nums))
ax.set_ylabel("Neurons (Inner) / Layers (Outer)", labelpad=20, fontweight='bold', fontsize=12)

for i, l in enumerate(layer_nums):
    center_y = (i * len(neuron_nums)) + (len(neuron_nums) - 1) / 2
    ax.text(-1.8, center_y, f"Lay={l}", ha='right', va='center', fontsize=11, fontweight='bold', color='navy')
    if i > 0:
        ax.axhline(y=(i * len(neuron_nums)) - 0.5, color='white', linewidth=2)

plt.title("SNN Hyperparameter Grid Search — F1 Score (IoU@0.5)", pad=50, fontsize=14)
plt.tight_layout()
plt.subplots_adjust(left=0.15, bottom=0.15)

plt.show()